In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/transit/bronze/landing/static"

TABLES = {
    "gtfs_stops":          "stops.txt",
    "gtfs_routes":         "routes.txt",
    "gtfs_trips":          "trips.txt",
    "gtfs_stop_times":     "stop_times.txt",
    "gtfs_calendar":       "calendar.txt",
    "gtfs_calendar_dates": "calendar_dates.txt",
}

# newest feed_version directory already on disk
dirs = sorted(d.name.rstrip("/") for d in dbutils.fs.ls(BASE))
out_dir = f"{BASE}/{dirs[-1]}"
feed_version = dirs[-1].split("=")[1]

agency_feed_version = (spark.read
    .option("header", True).option("inferSchema", False)
    .option("quote", '"').option("escape", '"')
    .csv(f"{out_dir}/feed_info.txt")
    .collect()[0]["feed_version"])

def read_gtfs(filename):
    """One GTFS file -> bronze-shaped DataFrame. All columns string."""
    return (spark.read
              .option("header", True)
              .option("inferSchema", False)
              .option("quote", '"')
              .option("escape", '"')
              .csv(f"{out_dir}/{filename}")
            .withColumn("_ingested_at",         F.current_timestamp())
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_feed_version",        F.lit(feed_version))
            .withColumn("_agency_feed_version", F.lit(agency_feed_version)))

def counts():
    return {t: spark.table(f"transit.bronze.{t}").count() for t in TABLES}

print("available versions:", dirs)
print("using:             ", feed_version)
print("agency string:     ", repr(agency_feed_version))

In [0]:
spark.sql("DROP TABLE IF EXISTS transit.bronze._append_demo")

df = read_gtfs("stops.txt")
df.write.format("delta").mode("append").saveAsTable("transit.bronze._append_demo")
print("after 1 append: ", f"{spark.table('transit.bronze._append_demo').count():,}")

df.write.format("delta").mode("append").saveAsTable("transit.bronze._append_demo")
print("after 2 appends:", f"{spark.table('transit.bronze._append_demo').count():,}")

spark.sql("DROP TABLE transit.bronze._append_demo")

In [0]:
for table, filename in TABLES.items():
    (read_gtfs(filename).write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .partitionBy("_feed_version")
       .saveAsTable(f"transit.bronze.{table}"))
    print(f"{table:22} partitioned")

ONE-TIME MIGRATION — already run. Do not re-run.

In [0]:
for table, filename in TABLES.items():
    (read_gtfs(filename).write
       .format("delta").mode("overwrite")
       .option("replaceWhere", f"_feed_version = '{feed_version}'")
       .saveAsTable(f"transit.bronze.{table}"))

for t, n in counts().items():
    print(f"{t:22} {n:>12,}")

In [0]:
for i in range(3):
    for table, filename in TABLES.items():
        (read_gtfs(filename).write
           .format("delta")
           .mode("overwrite")
           .option("replaceWhere", f"_feed_version = '{feed_version}'")
           .saveAsTable(f"transit.bronze.{table}"))
    print(f"run {i+1}:", counts())

In [0]:
spark.sql("""
  SELECT _feed_version, count(*) AS rows
  FROM transit.bronze.gtfs_stops
  GROUP BY _feed_version ORDER BY _feed_version
""").display()

spark.sql("DESCRIBE DETAIL transit.bronze.gtfs_stops").select(
    "numFiles", "sizeInBytes", "partitionColumns").display()

In [0]:
import re
from collections import Counter
from datetime import datetime, timezone

RT = "/Volumes/transit/bronze/landing/rt/vehicle_positions"
today = datetime.now(timezone.utc).strftime("%Y-%m-%d")

files = dbutils.fs.ls(f"{RT}/dt={today}")
stamps = [re.search(r"snapshot_(\d+)\.json", f.name).group(1) for f in files]
dupes = [s for s, n in Counter(stamps).items() if n > 1]

print(f"{len(files)} files · {len(set(stamps))} distinct timestamps · {len(dupes)} duplicates")